# Controlled ensemble experiment

Проверяем, может ли смесь двух model families улучшить weighted tuned XGBoost. Единица прогноза — `(route, date, hour)`, target — `boardings`. Более слабая standalone модель может исправлять ошибки сильной, если их residuals различаются. Это проверяется на исторических данных, а не предполагается заранее.

## Frozen contract и временная проверка

Используем tuned XGBoost и tuned LightGBM с одинаковым `SET_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURES`, параметрами из HPO и exponential recency weighting с half-life 180 дней. Для каждого fold вес считается от `max(fold_train.date)`. Фолды: Jan–Apr → May–Jun, Jan–Jun → Jul–Aug, Jan–Aug → Sep–Oct. Nov–Dec leaderboard не участвует в выборе alpha: это будущий период, и подбор по его score внёс бы повторное использование holdout.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ml/src/ensemble_experiment.py').exists())
sys.path.insert(0, str(root / 'ml/src'))
import ensemble_experiment as ee
import model_selection as ms

display(pd.DataFrame([{'fold': f[0], 'train': f'{f[1]}–{f[2]}', 'validation': f'{f[3]}–{f[4]}'} for f in ms.FOLDS]))

,fold,train,validation
0,jan_apr_to_may_jun,2025-01-01–2025-04-30,2025-05-01–2025-06-30
1,jan_jun_to_jul_aug,2025-01-01–2025-06-30,2025-07-01–2025-08-31
2,jan_aug_to_sep_oct,2025-01-01–2025-08-31,2025-09-01–2025-10-31


## Обучение и standalone references

Общая логика в `ml/src/ensemble_experiment.py` воспроизводит шесть frozen fits, сохраняет raw OOF predictions и проверяет точные fold и route absolute errors обеих weighted моделей против результатов recency experiments. При расхождении эксперимент останавливается до оценки blends.

In [2]:
result = ee.run_experiment()
print('Reference checks:', result['reference_checks'])
print('Raw predictions:', result['raw_predictions_path'])
print('JSON:', ee.ARTIFACT.relative_to(root))

Reference checks: Exact fold and per-route absolute errors reproduce prior weighted D results
Raw predictions: ml/experiments/results/ensemble_experiment_raw_predictions.csv.gz
JSON: ml\experiments\results\ensemble_experiment.json


## Complementarity ошибок

Сравниваем корреляции raw predictions, residuals и абсолютных ошибок, доли строк с меньшей ошибкой каждой модели и абсолютную разницу между прогнозами, включая разрез по маршрутам. Диагностика показывает потенциальную пользу смеси, но сама не служит целью оптимизации.

In [3]:
diagnostics = result['complementarity']
display(pd.DataFrame([{'fold': fold, **{k: v for k, v in item.items() if not isinstance(v, dict)}} for fold, item in diagnostics.items()]).round(6))
display(pd.DataFrame({fold: item['per_route_mean_absolute_prediction_difference'] for fold, item in diagnostics.items()}).T.round(3))

,fold,prediction_correlation,residual_correlation,absolute_error_correlation,xgb_lower_absolute_error_share,lgbm_lower_absolute_error_share,equal_absolute_error_share,mean_absolute_prediction_difference,median_absolute_prediction_difference
0,jan_apr_to_may_jun,0.996562,0.949353,0.942338,0.430328,0.569672,0.000000,28.048711,9.228165
1,jan_jun_to_jul_aug,0.999472,0.991148,0.989788,0.528629,0.471237,0.000134,17.117604,7.922240
2,jan_aug_to_sep_oct,0.999330,0.979715,0.974186,0.552937,0.446790,0.000273,19.723556,7.509475
3,pooled,0.998431,0.970715,0.965661,0.504099,0.495765,0.000136,21.605434,8.193894


,1,5,7,11,12,17,25,26,28,50
jan_apr_to_may_jun,16.779,0.304,24.971,33.569,37.426,102.590,11.573,20.978,10.971,21.326
jan_jun_to_jul_aug,12.265,0.709,20.498,18.132,23.320,43.284,10.090,13.927,11.499,17.453
jan_aug_to_sep_oct,13.072,2.446,25.966,25.096,24.525,45.172,7.250,19.277,9.064,25.367
pooled,14.029,1.151,23.793,25.558,28.396,63.571,9.640,18.038,10.517,21.361


## Fixed raw blends и общий postprocessing

Проверяем только `alpha = 1.00, 0.90, 0.75, 0.60, 0.50, 0.00` в формуле `alpha·XGB_raw + (1−alpha)·LGBM_raw`. Веса зафиксированы до оценки, чтобы не превращать этот шаг в HPO коэффициента. Смешивание идёт на raw predictions, затем один раз выполняются clipping, округление и правило `route 5 = 0`; смешение уже округлённых прогнозов меняло бы результат из-за нелинейного округления.

In [4]:
reference = result['summaries']['1.0']['pooled_wape']
table = pd.DataFrame([{'XGB weight': alpha, 'LGBM weight': 1-alpha, 'May–Jun': result['fold_metrics'][str(alpha)][0]['wape'], 'Jul–Aug': result['fold_metrics'][str(alpha)][1]['wape'], 'Sep–Oct': result['fold_metrics'][str(alpha)][2]['wape'], 'Pooled WAPE': result['summaries'][str(alpha)]['pooled_wape'], 'Δ vs XGB': result['summaries'][str(alpha)]['pooled_wape']-reference, 'Pooled AE': result['summaries'][str(alpha)]['pooled_absolute_error'], 'WAPE-score': result['summaries'][str(alpha)]['wape_score'], 'Mean fold WAPE': result['summaries'][str(alpha)]['mean_wape'], 'Median fold WAPE': result['summaries'][str(alpha)]['median_wape']} for alpha in ee.ALPHAS])
display(table.round(6))

,XGB weight,LGBM weight,May–Jun,Jul–Aug,Sep–Oct,Pooled WAPE,Δ vs XGB,Pooled AE,WAPE-score,Mean fold WAPE,Median fold WAPE
0,1.00,0.00,0.156928,0.198218,0.118587,0.155601,0.000000,5450387,0.844399,0.157911,0.156928
1,0.90,0.10,0.155299,0.198309,0.118184,0.154945,-0.000656,5427408,0.845055,0.157264,0.155299
2,0.75,0.25,0.152959,0.198499,0.117664,0.154041,-0.001559,5395768,0.845959,0.156374,0.152959
3,0.60,0.40,0.150814,0.198773,0.117259,0.153270,-0.002331,5368745,0.846730,0.155615,0.150814
4,0.50,0.50,0.149500,0.199025,0.117072,0.152846,-0.002755,5353884,0.847154,0.155199,0.149500
5,0.00,1.00,0.144429,0.200580,0.116781,0.151542,-0.004058,5308233,0.848458,0.153930,0.144429


## Стабильность по фолдам и маршрутам

Главный критерий — pooled WAPE: суммарная абсолютная ошибка, делённая на суммарный фактический поток всех трёх фолдов. Средний fold WAPE отвечает на другой вопрос, поскольку даёт периодам одинаковый вес. Поэтому также смотрим, не сосредоточен ли выигрыш в одном fold и нет ли сильного ухудшения маршрута. У route 5 WAPE не определён: его target и prediction равны нулю.

In [5]:
best = str(result['best_blend_alpha'])
routes = pd.DataFrame([{'route': route, 'XGB WAPE': result['summaries']['1.0']['per_route'][route]['wape'], 'Best blend WAPE': result['summaries'][best]['per_route'][route]['wape'], 'Δ WAPE': result['best_route_delta_pooled_wape_vs_xgb'].get(route)} for route in result['summaries']['1.0']['per_route']])
display(routes.round(6))
display(pd.DataFrame([{'fold': fold, 'Δ WAPE vs XGB': delta, 'result': 'gain' if delta < 0 else 'loss' if delta > 0 else 'tie'} for fold, delta in result['best_fold_delta_wape_vs_xgb'].items()]).round(6))
print('Classification:', result['classification'])

,route,XGB WAPE,Best blend WAPE,Δ WAPE
0,1,0.124341,0.124109,-0.000231
1,5,NaN,NaN,NaN
2,7,0.273041,0.272117,-0.000924
3,11,0.127854,0.125980,-0.001875
4,12,0.119222,0.118290,-0.000932
5,17,0.117136,0.111658,-0.005478
6,25,0.194396,0.191951,-0.002445
7,26,0.171733,0.166836,-0.004897
8,28,0.184639,0.181892,-0.002747
9,50,0.216200,0.213405,-0.002795


,fold,Δ WAPE vs XGB,result
0,jan_apr_to_may_jun,-0.007427,gain
1,jan_jun_to_jul_aug,0.000807,loss
2,jan_aug_to_sep_oct,-0.001515,gain


Classification: ENSEMBLE_SUPPORTED
